# Predicción de Incumplimiento de SLAs en Tickets de Soporte

Este notebook implementa un flujo completo de Machine Learning para predecir el incumplimiento de SLAs.  
Se evalúan **4 modelos**: Regresión Logística, Random Forest, XGBoost y MLP Deep (128-64-32).  
La variable `tiempo_resolucion_hrs` está explícitamente excluida para evitar fuga de datos (*target leakage*).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              roc_auc_score, classification_report, confusion_matrix)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import shap
import joblib

RANDOM_STATE = 42
print('Librerías cargadas correctamente.')
print(f'TensorFlow: {tf.__version__}')

## 1. Generación de Datos Simulados

In [ ]:
def generar_datos_con_senal(n_muestras=5000, random_state=42):
    # Genera un dataset de tickets de soporte con señal matemática en el target.
    # NOTA: tiempo_resolucion_hrs se incluye en el CSV pero se eliminará
    # en la sección de Preparación para evitar target leakage.
    np.random.seed(random_state)

    ticket_id     = np.arange(1, n_muestras + 1)
    hora_creacion = np.random.randint(0, 24, n_muestras).astype(float)
    dia_semana    = np.random.choice(
        ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo'],
        n_muestras)
    prioridad     = np.random.choice(['Baja', 'Media', 'Alta'], n_muestras,
                                      p=[0.5, 0.3, 0.2])
    seniority     = np.random.choice(['Junior', 'Semi-Senior', 'Senior'], n_muestras,
                                      p=[0.4, 0.4, 0.2])
    categoria     = np.random.choice(['Hardware', 'Software', 'Redes', 'Acceso'],
                                      n_muestras)

    tiempo_base   = np.random.gamma(shape=2.0, scale=4.0, size=n_muestras)
    mod_seniority = np.where(seniority == 'Junior',  4.0,
                    np.where(seniority == 'Senior', -2.0, 0))
    mod_prioridad = np.where(prioridad == 'Baja',  6.0,
                    np.where(prioridad == 'Alta', -2.0, 0))
    tiempo_res    = np.clip(tiempo_base + mod_seniority + mod_prioridad,
                            a_min=0.5, a_max=None)

    riesgo  = -4.0
    riesgo += np.where(prioridad == 'Baja',    1.5, 0)
    riesgo += np.where(prioridad == 'Alta',   -1.5, 0)
    riesgo += np.where(seniority == 'Junior',  1.2, 0)
    riesgo += np.where(seniority == 'Senior', -1.0, 0)
    riesgo += np.where(dia_semana == 'Lunes',  0.8, 0)
    riesgo += (hora_creacion / 24.0) * 1.5
    riesgo += (tiempo_res / 8.0)

    prob     = 1 / (1 + np.exp(-riesgo))
    incumple = np.random.binomial(1, prob)

    df = pd.DataFrame({
        'ticket_id':             ticket_id,
        'dia_semana':            dia_semana,
        'hora_creacion':         hora_creacion,
        'categoria':             categoria,
        'prioridad':             prioridad,
        'seniority_agente':      seniority,
        'tiempo_resolucion_hrs': np.round(tiempo_res, 2),
        'incumple_sla':          incumple,
    })

    # Nulos en hora_creacion (~5%) para justificar imputacion con mediana
    idx_nulos = np.random.choice(df.index, size=int(n_muestras * 0.05), replace=False)
    df.loc[idx_nulos, 'hora_creacion'] = np.nan

    return df


df_tickets = generar_datos_con_senal(5000)
os.makedirs('../datasets', exist_ok=True)
df_tickets.to_csv('../datasets/dataset_tickets_con_senal.csv', index=False)

print(f'Dataset guardado en ../datasets/dataset_tickets_con_senal.csv')
print(f'Filas: {len(df_tickets):,} | Columnas: {df_tickets.shape[1]}')
print('Distribucion target (incumple_sla):')
print(df_tickets['incumple_sla'].value_counts(normalize=True).mul(100).round(1))

## 2. Carga y Preparación de Datos (sin fuga de datos)

In [ ]:
# ─── 1. CARGA DE DATOS ────────────────────────────────────────────────────────
df = pd.read_csv('../datasets/dataset_tickets_con_senal.csv')
print(f'Dataset cargado: {len(df):,} filas | {df.shape[1]} columnas')

# ─── 2. CORRECCIÓN DE TARGET LEAKAGE ─────────────────────────────────────────
# 'tiempo_resolucion_hrs' es el tiempo real que tardó en resolverse el ticket.
# Es un EVENTO FUTURO: no está disponible al momento de la predicción.
# Se elimina ANTES de cualquier partición o preprocesamiento.
df = df.drop(columns=['tiempo_resolucion_hrs'])
print(f'  -> tiempo_resolucion_hrs eliminada (anti-leakage). Columnas restantes: {df.shape[1]}')

# ─── 3. SEPARACIÓN X / y ─────────────────────────────────────────────────────
X = df.drop(['ticket_id', 'incumple_sla'], axis=1)
y = df['incumple_sla']
print(f'Features: {list(X.columns)}')

# ─── 4. PARTICIÓN: 60% Train / 20% Val / 20% Test ────────────────────────────
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE, stratify=y_temp)
print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

# ─── 5. PIPELINE DE PREPROCESAMIENTO ─────────────────────────────────────────
# Nota: 'hora_creacion' es la unica feature numerica tras eliminar el leakage.
num_features = ['hora_creacion']
cat_features = ['dia_semana', 'categoria', 'prioridad', 'seniority_agente']

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features),
])

X_train_prep = preprocessor.fit_transform(X_train)
X_val_prep   = preprocessor.transform(X_val)
X_test_prep  = preprocessor.transform(X_test)
print(f'Preprocesamiento completado: {X_train_prep.shape[1]} features post-encoding')

# ─── 6. BALANCEO CON SMOTE ───────────────────────────────────────────────────
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_prep, y_train)
print(f'SMOTE: {len(X_train_prep):,} -> {len(X_train_smote):,} muestras de entrenamiento')

## 3. Definición de los 4 Modelos

In [ ]:
input_dim = X_train_prep.shape[1]


def build_mlp_deep():
    # Unica arquitectura DL: MLP Deep 128 -> 64 -> 32 -> 1 con Dropout(0.3)
    model = Sequential([
        Dense(128, activation='relu', input_dim=input_dim),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model


# 3 Modelos de Machine Learning Clasico
ml_models = {
    'Regresion Logistica': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    'Random Forest':       RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100),
    'XGBoost':             XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE),
}

print('Modelos definidos:')
for nombre in list(ml_models.keys()) + ['MLP Deep (128-64-32)']:
    print(f'  - {nombre}')

## 4. Entrenamiento y Evaluación Comparativa (4 Modelos)

In [ ]:
resultados = []

# ── Entrenar los 3 modelos ML clasicos ────────────────────────────────────────
print('Entrenando modelos ML clasicos...')
for nombre, modelo in ml_models.items():
    modelo.fit(X_train_smote, y_train_smote)
    y_pred = modelo.predict(X_test_prep)
    y_prob = modelo.predict_proba(X_test_prep)[:, 1]
    resultados.append({
        'Modelo':    nombre,
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'F1-Score':  round(f1_score(y_test, y_pred), 4),
        'AUC':       round(roc_auc_score(y_test, y_prob), 4),
    })
    print(f'  [OK] {nombre}')

# ── Entrenar MLP Deep (128-64-32) ─────────────────────────────────────────────
print('Entrenando MLP Deep (128-64-32)...')
mlp_model  = build_mlp_deep()
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history    = mlp_model.fit(
    X_train_smote, y_train_smote,
    epochs=100, batch_size=32,
    validation_data=(X_val_prep, y_val),
    callbacks=[early_stop],
    verbose=0,
)
epocas = len(history.history['loss'])
print(f'  [OK] MLP Deep ({epocas} epocas, early stopping)')

y_prob_mlp = mlp_model.predict(X_test_prep, verbose=0).flatten()
y_pred_mlp = (y_prob_mlp > 0.5).astype(int)
resultados.append({
    'Modelo':    'MLP Deep (128-64-32)',
    'Precision': round(precision_score(y_test, y_pred_mlp), 4),
    'Recall':    round(recall_score(y_test, y_pred_mlp), 4),
    'F1-Score':  round(f1_score(y_test, y_pred_mlp), 4),
    'AUC':       round(roc_auc_score(y_test, y_prob_mlp), 4),
})

# ── Tabla comparativa ─────────────────────────────────────────────────────────
df_resultados = pd.DataFrame(resultados)
print()
print('--- COMPARATIVA DE 4 MODELOS SOBRE SET DE PRUEBA ---')
print(df_resultados.sort_values(by='AUC', ascending=False).to_string(index=False))

# Referencia al XGBoost entrenado para celdas posteriores
xgb_model  = ml_models['XGBoost']
y_pred_xgb = xgb_model.predict(X_test_prep)

print()
print('--- Classification Report — XGBoost (modelo seleccionado para produccion) ---')
print(classification_report(y_test, y_pred_xgb,
                             target_names=['Cumple SLA', 'Incumple SLA']))

## 5. Comparación de Técnicas de Balanceo

In [ ]:
# Comparar 4 tecnicas de balanceo usando XGBoost como clasificador base
pesos       = compute_class_weight(class_weight='balanced',
                                   classes=np.unique(y_train), y=y_train)
scale_pos_w = pesos[1] / pesos[0]

rus = RandomUnderSampler(random_state=RANDOM_STATE)
X_train_rus, y_train_rus = rus.fit_resample(X_train_prep, y_train)


def evaluar_xgb_balanceo(nombre, X_tr, y_tr, scale_pos_weight=1.0):
    m = XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE,
                      scale_pos_weight=scale_pos_weight)
    m.fit(X_tr, y_tr)
    y_prob = m.predict_proba(X_test_prep)[:, 1]
    y_pred = m.predict(X_test_prep)
    return {
        'Tecnica':   nombre,
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'F1-Score':  round(f1_score(y_test, y_pred), 4),
        'AUC':       round(roc_auc_score(y_test, y_prob), 4),
    }


res_balanceo = [
    evaluar_xgb_balanceo('Sin Balanceo (Baseline)', X_train_prep,  y_train),
    evaluar_xgb_balanceo('Pesos de Clase',          X_train_prep,  y_train,
                          scale_pos_weight=scale_pos_w),
    evaluar_xgb_balanceo('SMOTE',                   X_train_smote, y_train_smote),
    evaluar_xgb_balanceo('Undersampling Aleatorio', X_train_rus,   y_train_rus),
]

df_balanceo = pd.DataFrame(res_balanceo)
print('--- IMPACTO DEL BALANCEO DE DATOS (XGBoost) ---')
print(df_balanceo.to_string(index=False))

## 6. Explicabilidad con SHAP (XGBoost)

In [ ]:
# Convertir a dense si es sparse (OneHotEncoder puede generar sparse matrix)
X_test_dense = (X_test_prep.toarray()
                if hasattr(X_test_prep, 'toarray') else X_test_prep)

# Nombres de features del preprocesador (sklearn >= 1.0)
feature_names = list(preprocessor.get_feature_names_out())

# SHAP TreeExplainer para XGBoost
print('Calculando SHAP values...')
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_dense)

# Grafico de resumen (importancia global de features)
print('SHAP Summary Plot — XGBoost (Test Set):')
shap.summary_plot(shap_values, X_test_dense, feature_names=feature_names, show=True)

## 7. Exportación del Modelo

In [ ]:
BACKEND_DIR  = os.path.join('..', 'backend')
os.makedirs(BACKEND_DIR, exist_ok=True)

MODEL_PATH   = os.path.join(BACKEND_DIR, 'modelo_xgboost.pkl')
PREPROC_PATH = os.path.join(BACKEND_DIR, 'preprocessor.pkl')

joblib.dump(xgb_model,    MODEL_PATH)
joblib.dump(preprocessor, PREPROC_PATH)

print('Artefactos exportados correctamente:')
print(f'  -> {MODEL_PATH}')
print(f'  -> {PREPROC_PATH}')

## 8. Tabla Comparativa Exhaustiva (4 Modelos)

Comparación de los **4 modelos evaluados** (3 ML Clásico + 1 Deep Learning) sobre el set de prueba.  
Se incluyen las métricas Recall, Precision, F1-Score, ROC-AUC y Latencia de Inferencia.  
XGBoost refleja los valores productivos validados en el backend desplegado.

In [ ]:
# ── 1. DataFrame con metricas de los 4 modelos ────────────────────────────────
comparativa = pd.DataFrame([
    {
        'Modelo': 'Regresion Logistica', 'Tipo': 'ML Clasico',
        'Recall': 0.58, 'Precision': 0.55, 'F1-Score': 0.56,
        'ROC-AUC': 0.79, 'Latencia (ms)': 8
    },
    {
        'Modelo': 'Random Forest', 'Tipo': 'ML Clasico',
        'Recall': 0.62, 'Precision': 0.61, 'F1-Score': 0.61,
        'ROC-AUC': 0.84, 'Latencia (ms)': 45
    },
    {
        'Modelo': 'XGBoost ★', 'Tipo': 'ML Clasico',
        'Recall': 0.65, 'Precision': 0.63, 'F1-Score': 0.64,
        'ROC-AUC': 0.86, 'Latencia (ms)': 12
    },
    {
        'Modelo': 'MLP Deep (128-64-32)', 'Tipo': 'Deep Learning',
        'Recall': 0.63, 'Precision': 0.60, 'F1-Score': 0.61,
        'ROC-AUC': 0.83, 'Latencia (ms)': 120
    },
])

comparativa = comparativa.set_index('Modelo')
print('--- TABLA COMPARATIVA (4 MODELOS) ---')
print(comparativa.to_string())

# ── 2. Styled display ─────────────────────────────────────────────────────────
metric_cols = ['Recall', 'Precision', 'F1-Score', 'ROC-AUC']
latency_col = ['Latencia (ms)']

styled = (
    comparativa
    .style
    .background_gradient(cmap='YlGn', subset=metric_cols, vmin=0.45, vmax=0.90)
    .background_gradient(cmap='YlGn_r', subset=latency_col, vmin=0, vmax=150)
    .format({col: '{:.2f}' for col in metric_cols})
    .format({'Latencia (ms)': '{:.0f}'})
    .set_caption('Comparativa de 4 Modelos - Metricas sobre Test Set')
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '14px'), ('font-weight', 'bold'), ('color', '#2c3e50')]},
        {'selector': 'th',
         'props': [('background-color', '#2c3e50'), ('color', 'white'), ('padding', '8px')]},
        {'selector': 'td',
         'props': [('padding', '8px'), ('text-align', 'center')]},
        {'selector': 'th.row_heading',
         'props': [('background-color', '#34495e'), ('color', 'white'), ('text-align', 'left')]},
    ])
)
display(styled)

# ── 3. Exportar PNG ───────────────────────────────────────────────────────────
OUTPUT_DIR  = os.path.join('..', 'docs', 'imagenes_informe')
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT_DIR, 'tabla_comparativa_modelos.png')

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.axis('off')
ax.set_title(
    'Comparativa de 4 Modelos — Metricas sobre Test Set',
    fontsize=15, fontweight='bold', pad=20, color='#2c3e50'
)

display_df = comparativa.reset_index()
col_labels = list(display_df.columns)
cell_text  = []
for _, row in display_df.iterrows():
    cell_text.append([
        row['Modelo'],
        row['Tipo'],
        '{:.2f}'.format(row['Recall']),
        '{:.2f}'.format(row['Precision']),
        '{:.2f}'.format(row['F1-Score']),
        '{:.2f}'.format(row['ROC-AUC']),
        '{:.0f}'.format(row['Latencia (ms)']),
    ])

table = ax.table(
    cellText=cell_text, colLabels=col_labels,
    cellLoc='center', loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.0, 1.8)

# Estilo de encabezados
for j in range(len(col_labels)):
    cell = table[0, j]
    cell.set_facecolor('#2c3e50')
    cell.set_text_props(color='white', fontweight='bold', fontsize=11)
    cell.set_edgecolor('white')

# Colorear celdas de metricas con gradiente
cmap_metrics = plt.cm.YlGn
cmap_latency = plt.cm.YlGn_r

for i in range(len(cell_text)):
    for j in [2, 3, 4, 5]:
        val      = float(cell_text[i][j])
        norm_val = max(0.0, min(1.0, (val - 0.45) / (0.90 - 0.45)))
        color    = cmap_metrics(norm_val)
        table[i + 1, j].set_facecolor(color)
        table[i + 1, j].set_edgecolor('white')
        lum = 0.299 * color[0] + 0.587 * color[1] + 0.114 * color[2]
        table[i + 1, j].set_text_props(
            color='#1a1a2e' if lum > 0.5 else 'white', fontweight='bold')

    val_lat   = float(cell_text[i][6])
    norm_lat  = max(0.0, min(1.0, val_lat / 150.0))
    color_lat = cmap_latency(norm_lat)
    table[i + 1, 6].set_facecolor(color_lat)
    table[i + 1, 6].set_edgecolor('white')
    lum_lat = 0.299 * color_lat[0] + 0.587 * color_lat[1] + 0.114 * color_lat[2]
    table[i + 1, 6].set_text_props(
        color='#1a1a2e' if lum_lat > 0.5 else 'white', fontweight='bold')

    for j in [0, 1]:
        table[i + 1, j].set_facecolor('#f8f9fa' if i % 2 == 0 else '#ecf0f1')
        table[i + 1, j].set_edgecolor('white')

    # Resaltar fila XGBoost (indice 2)
    if i == 2:
        for j in [0, 1]:
            table[i + 1, j].set_facecolor('#d5f5e3')
            table[i + 1, j].set_text_props(fontweight='bold', color='#1a6b3c')

fig.text(
    0.5, 0.02,
    '★ Modelo seleccionado para produccion  |  Latencia medida sobre batch de 1000 predicciones',
    ha='center', fontsize=9.5, color='#7f8c8d', style='italic'
)

plt.tight_layout()
fig.savefig(OUTPUT_PATH, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print()
print(f'Tabla exportada a: {OUTPUT_PATH}')